# Day 54 — Boosting + cost-tuned threshold

Status: COMPLETE. XGBoost vs LightGBM (300 trees) on a 64/16/20 split;
threshold = argmin *validation* cost (never tuned on test). Winner by val $:
XGBoost. 12/12 tests pass.

In [1]:
import json
from pathlib import Path

m = json.loads(Path("../models/tuning_metrics.json").read_text())["models"]
for name in ("xgboost", "lightgbm"):
    v = m[name]
    print(f"{name:<9}: ROC {v['roc_auc']} PR {v['pr_auc']} | thr {v['best_threshold']} | "
          f"val ${v['val_cost']:,.0f} | test ${v['test_cost']['total']:,.0f}")
print("vs baseline: ROC 0.68 -> 0.73, cost@0.5 $301k -> cost-tuned ~$90k (3.4x)")
print("read: threshold did most of the work (0.5 -> 0.15); model choice is second.")

xgboost : ROC 0.7339 PR 0.3977 | thr 0.15 | val $67,650 | test $90,135
lightgbm: ROC 0.7310 PR 0.3964 | thr 0.10 | val $69,035 | test $88,275
vs baseline: ROC 0.68 -> 0.73, cost@0.5 $301k -> cost-tuned ~$90k (3.4x)
read: threshold did most of the work (0.5 -> 0.15); model choice is second.


## Findings

1. **The threshold beat the model**: $301k → ~$90k came from 0.5 → 0.15,
   while XGBoost-vs-LightGBM differs by 2% on val cost. Tune decisions before
   chasing architectures.
2. **Tuned on val, reported on test**: threshold never saw the test set — the
   $88–90k figures are honest estimates, not overfit ones.
3. **Winner = cheapest on val (XGBoost)**, the production pick from here on.